# Librerias y Carga de datos.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# 1. Configuración de Pandas
pd.set_option('display.max_columns', None)


# 2. Carga de datos maestros y encuestas (Factores demográficos y de estrés).
df_general = pd.read_csv('../data/raw/general_data.csv')
df_employee = pd.read_csv('../data/raw/employee_survey_data.csv')
df_manager = pd.read_csv('../data/raw/manager_survey_data.csv')

# 3. Carga de registros horarios brutos.
# Nota: Pasamos low_memory=False porque son archivos masivos con muchas fechas.
df_in = pd.read_csv('../data/raw/in_time.csv', low_memory=False)
df_out = pd.read_csv('../data/raw/out_time.csv', low_memory=False)

print(f"✔ Datos maestros cargados: {df_general.shape[0]} empleados.")
print(f"✔ Registros de entrada 'In' detectados: {df_in.shape[0]} filas y {df_in.shape[1]} columnas (fechas).")

✔ Datos maestros cargados: 4410 empleados.
✔ Registros de entrada 'In' detectados: 4410 filas y 262 columnas (fechas).


# Días de Absentismo.

In [3]:
# Convertimos la primera columna (sin nombre) en el ID de empleado.
df_in.rename(columns={df_in.columns[0]: 'EmployeeID'}, inplace=True)
df_out.rename(columns={df_out.columns[0]: 'EmployeeID'}, inplace=True)

# Las columnas de fechas son todas excepto 'EmployeeID'.
columnas_fechas = [col for col in df_in.columns if col != 'EmployeeID']

# Contamos cuántos valores nulos (NaN) tiene cada empleado en las columnas de fechas.
# Un valor nulo significa que ese día el empleado NO fichó la entrada (Absentismo).
df_general['Dias_Absentismo'] = df_in[columnas_fechas].isnull().sum(axis=1)

# Inspeccionamos rápidamente las estadísticas de nuestra nueva variable objetivo.
print("--- VARIABLE OBJETIVO CALCULADA ---")
print(df_general['Dias_Absentismo'].describe())

--- VARIABLE OBJETIVO CALCULADA ---
count    4410.000000
mean       24.734694
std         5.503779
min        13.000000
25%        20.000000
50%        25.000000
75%        29.000000
max        36.000000
Name: Dias_Absentismo, dtype: float64


# Encuestas Empleados y Manager.

In [4]:
# Imprimimos las columnas de la encuesta del empleado.
print("--- COLUMNAS EN EMPLOYEE SURVEY ---")
print(df_employee.columns.tolist())
print("\n--- EJEMPLO DE DATOS ---")
print(df_employee.head(5))

print("\n" + "="*40 + "\n")

# Imprimimos las columnas de la encuesta del manager.
print("--- COLUMNAS EN MANAGER SURVEY ---")
print(df_manager.columns.tolist())
print("\n--- EJEMPLO DE DATOS ---")
print(df_manager.head(5))

--- COLUMNAS EN EMPLOYEE SURVEY ---
['EmployeeID', 'EnvironmentSatisfaction', 'JobSatisfaction', 'WorkLifeBalance']

--- EJEMPLO DE DATOS ---
   EmployeeID  EnvironmentSatisfaction  JobSatisfaction  WorkLifeBalance
0           1                      3.0              4.0              2.0
1           2                      3.0              2.0              4.0
2           3                      2.0              2.0              1.0
3           4                      4.0              4.0              3.0
4           5                      4.0              1.0              3.0


--- COLUMNAS EN MANAGER SURVEY ---
['EmployeeID', 'JobInvolvement', 'PerformanceRating']

--- EJEMPLO DE DATOS ---
   EmployeeID  JobInvolvement  PerformanceRating
0           1               3                  3
1           2               2                  4
2           3               3                  3
3           4               2                  3
4           5               3                  3


# Relacionamos las encuestas con nuestro general_data

In [5]:
# 1.Unión limpia de los datos maestros con las encuestas.
df_general = pd.read_csv('../data/raw/general_data.csv')
df_general['Dias_Absentismo'] = df_in[columnas_fechas].isnull().sum(axis=1)

# --- CONVERSIÓN DE TEXTO A FECHAS Y CÁLCULO DE JORNADA ---
# Convertimos todas las columnas horarias a formato DateTime real.
df_in_dt = df_in[columnas_fechas].apply(pd.to_datetime, errors='coerce')
df_out_dt = df_out[columnas_fechas].apply(pd.to_datetime, errors='coerce')

# Restamos las horas (Salida - Entrada).
df_horas_diarias = df_out_dt - df_in_dt
df_horas_num = df_horas_diarias.apply(lambda x: x.dt.total_seconds() / 3600)
df_general['Media_Horas_Diarias'] = df_horas_num.mean(axis=1)
# --------------------------------------------------------

# Unimos las encuestas.
df_general = pd.merge(df_general, df_employee, on='EmployeeID', how='left')
df_general = pd.merge(df_general, df_manager, on='EmployeeID', how='left')

# 2. Separamos las columnas a la mitad para visualizarlas en paralelo.
todas_las_columnas = df_general.columns.tolist()
mitad = (len(todas_las_columnas) + 1) // 2
columna_izq = todas_las_columnas[:mitad]
columna_der = todas_las_columnas[mitad:]

# 3. Imprimimos las dos columnas limpias.
print(f"{'--- COLUMNAS PARTE 1 ---':<35} | {'--- COLUMNAS PARTE 2 ---':<35}")
for i in range(max(len(columna_izq), len(columna_der))):
    izq = columna_izq[i] if i < len(columna_izq) else ""
    der = columna_der[i] if i < len(columna_der) else ""
    print(f"- {izq:<33} | - {der:<33}")

--- COLUMNAS PARTE 1 ---            | --- COLUMNAS PARTE 2 ---           
- Age                               | - PercentSalaryHike                
- Attrition                         | - StandardHours                    
- BusinessTravel                    | - StockOptionLevel                 
- Department                        | - TotalWorkingYears                
- DistanceFromHome                  | - TrainingTimesLastYear            
- Education                         | - YearsAtCompany                   
- EducationField                    | - YearsSinceLastPromotion          
- EmployeeCount                     | - YearsWithCurrManager             
- EmployeeID                        | - Dias_Absentismo                  
- Gender                            | - Media_Horas_Diarias              
- JobLevel                          | - EnvironmentSatisfaction          
- JobRole                           | - JobSatisfaction                  
- MaritalStatus                     | 

# Inspección de valores.

In [6]:
# Contamos cuántos valores nulos hay en cada columna del dataset consolidado.
valores_nulos = df_general.isnull().sum()

# Filtramos para mostrar solo las columnas que SÍ tienen nulos.
columnas_con_nulos = valores_nulos[valores_nulos > 0]

print("--- RECUENTO DE VALORES NULOS EN EL DATASET ---")
if len(columnas_con_nulos) > 0:
    print(columnas_con_nulos)
else:
    print("¡Perfecto! No hay ningún valor nulo en todo el dataset.")

--- RECUENTO DE VALORES NULOS EN EL DATASET ---
NumCompaniesWorked         19
TotalWorkingYears           9
EnvironmentSatisfaction    25
JobSatisfaction            20
WorkLifeBalance            38
dtype: int64


# Imputación de valores nulos.

In [7]:
# Rellenamos las columnas numéricas demográficas con su mediana.
df_general['NumCompaniesWorked'] = df_general['NumCompaniesWorked'].fillna(df_general['NumCompaniesWorked'].median())
df_general['TotalWorkingYears'] = df_general['TotalWorkingYears'].fillna(df_general['TotalWorkingYears'].median())

# Rellenamos las encuestas de satisfacción con la moda (el valor que más se repite).
df_general['EnvironmentSatisfaction'] = df_general['EnvironmentSatisfaction'].fillna(df_general['EnvironmentSatisfaction'].mode()[0])
df_general['JobSatisfaction'] = df_general['JobSatisfaction'].fillna(df_general['JobSatisfaction'].mode()[0])
df_general['WorkLifeBalance'] = df_general['WorkLifeBalance'].fillna(df_general['WorkLifeBalance'].mode()[0])

# Verificamos que ya no quede ningún nulo.
print("--- VERIFICACIÓN FINAL DE NULOS ---")
print(f"Total de valores nulos pendientes: {df_general.isnull().sum().sum()}")

--- VERIFICACIÓN FINAL DE NULOS ---
Total de valores nulos pendientes: 0


In [11]:
import os

# 1. Creamos la carpeta de destino si no existe (para guardar el dataset limpio).
carpeta_destino = '../data/processed'
os.makedirs(carpeta_destino, exist_ok=True)

# 2. Guardamos el dataset unificado y limpio
df_general.to_csv(f'{carpeta_destino}/df_consolidado_clean.csv', index=False)

print("--- ¡PROCESO ETL FINALIZADO CON ÉXITO! ---")
print("La carpeta 'data/processed/' ha sido creada automáticamente.")
print("El archivo 'df_consolidado_clean.csv' se ha guardado correctamente ahí.")

--- ¡PROCESO ETL FINALIZADO CON ÉXITO! ---
La carpeta 'data/processed/' ha sido creada automáticamente.
El archivo 'df_consolidado_clean.csv' se ha guardado correctamente ahí.
